In [ ]:
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
import ta
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

In [ ]:
# ====================== 50K 帳戶嚴格設定 ======================
POINT_VALUE = {
    'MES.F': 5, 'MNQ.F': 2, 'M2K.F': 5, 'MYM.F': 0.5,
    'M6E.F': 12500, 'M6A.F': 10000, 'MCL.F': 100
}
CONTRACTS = {'MES.F':2, 'MNQ.F':2, 'M2K.F':2, 'MYM.F':2, 'M6E.F':2, 'M6A.F':2, 'MCL.F':1}  # MCL 只用 1 contract
PRECISION = {
    'MES.F':2, 'MNQ.F':2, 'M2K.F':2, 'MYM.F':2,
    'M6E.F':4, 'M6A.F':4, 'MCL.F':2
}
CSV_FILES = {k: f"{k}.csv" for k in POINT_VALUE.keys()}
SCAN_FILE = '7_Micro_15min_KillZone_LSTM_Auto.csv'
JOURNAL_FILE = 'daily_journal.csv'

def is_kill_zone(dt_utc):
    dt_est = dt_utc - timedelta(hours=4)
    h, m = dt_est.hour, dt_est.minute
    if 3 <= h < 4: return "London"
    if (h == 9 and m >= 30) or (h == 10 and m < 30): return "NY"
    return None

def run_lstm_scan():
    now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S HKT')
    print(f"\n{'='*70}")
    print(f"🕒 {now_str} LSTM 掃描開始")
    latest_signals = {}
    
    for sym, fname in CSV_FILES.items():
        try:
            print(f"  訓練 {sym}...", end='', flush=True)
            df = pd.read_csv(fname, parse_dates=['datetime'])
            df.set_index('datetime', inplace=True)
            df.sort_index(inplace=True)
            
            # ICT + TA features
            df['RSI'] = ta.RSIIndicator(df['close'], window=14).rsi()
            df['EMA12'] = ta.EMAIndicator(df['close'], window=12).ema_indicator()
            df['EMA26'] = ta.EMAIndicator(df['close'], window=26).ema_indicator()
            macd = ta.MACD(df['close'])
            df['MACD'] = macd.macd
            df['MACD_signal'] = macd.macd_signal
            bb = ta.BollingerBands(df['close'])
            df['BB_upper'] = bb.bollinger_hband()
            df['BB_middle'] = bb.bollinger_mavg()
            df['BB_lower'] = bb.bollinger_lband()
            df['ATR'] = ta.AverageTrueRange(df['high'], df['low'], df['close'], window=14).average_true_range()
            df['Volume_SMA'] = ta.SMAIndicator(df['volume'], window=20).sma_indicator()
            
            df['FVG_Bullish'] = (df['low'].shift(1) > df['high'].shift(2)).astype(int)
            df['FVG_Bearish'] = (df['high'].shift(1) < df['low'].shift(2)).astype(int)
            df['Swing_High'] = df['high'].rolling(20).max().shift(1)
            df['Swing_Low'] = df['low'].rolling(20).min().shift(1)
            df['Fib_Range'] = df['Swing_High'] - df['Swing_Low']
            df['OTE_079_Bull'] = df['Swing_Low'] + df['Fib_Range'] * 0.79
            df['OTE_079_Bear'] = df['Swing_High'] - df['Fib_Range'] * 0.79
            df['In_OTE_Bull'] = ((df['close'] >= df['Swing_Low'] + df['Fib_Range']*0.62) & (df['close'] <= df['OTE_079_Bull'])).astype(int)
            df['In_OTE_Bear'] = ((df['close'] <= df['Swing_High'] - df['Fib_Range']*0.62) & (df['close'] >= df['OTE_079_Bear'])).astype(int)
            df.dropna(inplace=True)
            
            # LSTM
            data = df.filter(['close'])
            dataset = data.values
            scaler = MinMaxScaler(feature_range=(0,1))
            scaled_data = scaler.fit_transform(dataset)
            train_size = int(len(scaled_data) * 0.95)
            train_data = scaled_data[:train_size]
            x_train, y_train = [], []
            for i in range(60, len(train_data)):
                x_train.append(train_data[i-60:i, 0])
                y_train.append(train_data[i, 0])
            x_train, y_train = np.array(x_train), np.array(y_train)
            x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
            
            model = Sequential([LSTM(128, return_sequences=True, input_shape=(x_train.shape[1], 1)),
                               LSTM(64, return_sequences=False),
                               Dense(25), Dense(1)])
            model.compile(optimizer='adam', loss='mean_squared_error')
            model.fit(x_train, y_train, batch_size=1, epochs=1, verbose=0)
            
            last_60 = scaled_data[-60:]
            last_60 = np.reshape(last_60, (1, 60, 1))
            pred = scaler.inverse_transform(model.predict(last_60, verbose=0))[0][0]
            
            latest = df.iloc[-1].copy()
            ml_signal = 1 if (pred > latest['close'] + latest['ATR']*0.3) else (-1 if pred < latest['close'] - latest['ATR']*0.3 else 0)
            trend_bull = 1 if latest['EMA12'] > latest['EMA26'] else 0
            
            hybrid = 0
            if ml_signal == 1 and trend_bull == 1 and (latest['FVG_Bullish'] or latest['In_OTE_Bull']):
                hybrid = 1
            elif ml_signal == -1 and trend_bull == 0 and (latest['FVG_Bearish'] or latest['In_OTE_Bear']):
                hybrid = -1
            
            kz = is_kill_zone(latest.name)
            pv = POINT_VALUE[sym]
            contracts = CONTRACTS[sym]
            prec = PRECISION[sym]
            est_pnl = round((pred - latest['close']) * pv * contracts if hybrid == 1 else (latest['close'] - pred) * pv * contracts, 2)
            
            latest_signals[sym] = {
                'Timestamp_EST': latest.name.strftime('%Y-%m-%d %H:%M EST'),
                'Close': round(latest['close'], prec),
                'Prediction': round(pred, prec),
                'Hybrid_Signal': hybrid,
                'In_KillZone': kz,
                'Est_PnL': est_pnl,
                'Contracts': contracts
            }
            print(f" ✅ Close={latest['close']} Signal={hybrid} KZ={kz}")
        except Exception as e:
            print(f" ❌ {sym}: {e}")
    
    summary = pd.DataFrame(latest_signals).T
    cols = ['Timestamp_EST', 'Close', 'Prediction', 'Hybrid_Signal', 'In_KillZone', 'Est_PnL', 'Contracts']
    print(f"\n{'='*70}")
    print(f"📊 LSTM 訊號總覽")
    print(f"{'='*70}")
    print(summary[cols].to_string())
    
    # Save scan results
    summary.to_csv(SCAN_FILE, mode='a', header=False)
    
    # Daily journal
    today = datetime.now().date()
    total_pnl = summary['Est_PnL'].sum()
    journal_entry = pd.DataFrame([{'Date': today, 'Total_PnL': round(total_pnl, 2), 'WinRate': 0}])
    journal_entry.to_csv(JOURNAL_FILE, mode='a', header=False, index=False)
    print(f"\n📊 daily_journal.csv 已更新 | Total Est PnL: ${round(total_pnl, 2)}")
    
    return summary

print("✅ 函數定義完成，運行 Cell 3 開始自動排程")

In [ ]:
# ====================== 自動排程 ======================
print("🚀 本機 LSTM 自動排程版已啟動（24小時無限跑）")
print("Profit Target $3,000 | 至少 5 個合格獲利日（每日 ≥ $250） | $200 Daily SL kill-switch")
print("MCL 提醒：只用 1 contract（2 contracts = $200/點）")
print("London KZ: 15:00-16:00 HKT | NY KZ: 21:30-22:30 HKT")
print("按 Ctrl+C 停止\n")

run_count = 0
while True:
    run_count += 1
    print(f"\n{'#'*70}")
    print(f"第 {run_count} 次掃描")
    summary = run_lstm_scan()
    
    in_zone = any(row['In_KillZone'] is not None for _, row in summary.iterrows())
    if in_zone:
        kz_rows = summary[summary['In_KillZone'].notna() & (summary['Hybrid_Signal'] != 0)]
        if len(kz_rows) > 0:
            print("\n🔥 🔥 🔥 KILL ZONE 內有訊號！立即檢查下單機會 🔥 🔥 🔥")
            for sym, row in kz_rows.iterrows():
                direction = "Long" if row['Hybrid_Signal'] == 1 else "Short"
                print(f"  → {sym}: {direction} @ {row['Close']} | Est ${row['Est_PnL']} | {row['Contracts']} contract(s)")
    else:
        print("\n⏳ 不在 Kill Zone，安全等待...")
    
    time.sleep(900)  # 15 分鐘